# MTCA-1 Stage 2: Corpus Assembly

**Study identifier:** MTCA-1 (Mixed-Tier Corpus Analysis, Study 1)
**Program:** IRMB / Hudson Forge Technologies LLC
**Researcher:** Billy Davis
**Stage:** 2 of 10 — Corpus Assembly
**Prior stage:** Stage 1 Foundational Document (frozen)

---

## Purpose

Assemble the validation corpus for MTCA-1. The corpus consists of five texts drawn from four distinct lineages, all exhibiting the content class defined in Section 2 of the Stage 1 Foundational Document.

## Corpus composition (locked)

| ID | Title | Author | Lineage | License | Source |
|----|-------|--------|---------|---------|--------|
| C1 | Self Mastery Through Conscious Autosuggestion | Émile Coué (1922) | Therapeutic-autosuggestion | Public domain | Project Gutenberg #27203 |
| C3 | Meditations (selected book) | Marcus Aurelius (~170 CE, George Long trans. 1862) | Stoic self-directed | Public domain | Project Gutenberg #2680 |
| C5 | As a Man Thinketh | James Allen (1903) | New Thought / metaphysical-aspirational | Public domain | Project Gutenberg #4507 |
| C7 | Twelve Steps (canonical text) | Anonymous (1939) | Behavioral commitment / recovery | Public domain | AA General Service literature |
| C8 | Synthetic CBT analogue | Authored for this study | Behavioral commitment / contemporary practical | Original work | Authored in this notebook |

Marina Tudor's framework is not in this corpus. Inclusion would require Stage 3 consent process. The corpus design does not depend on her inclusion.

## What this notebook does

1. Fetches public-domain texts from canonical sources
2. Extracts the relevant portions per text (no copyright concerns since all are public domain)
3. Segments each text into analyzable units
4. Authors the synthetic CBT analogue in-notebook for full transparency
5. Computes SHA256 hash of the assembled corpus
6. Saves the corpus in a frozen format for downstream stages

## What this notebook does NOT do

- No hand-labeling of units. Unit classification is performed by the council classifier at runtime in Stage 6.
- No ground-truth tier assignment. The study reports model behavior, not adjudication of text content.
- No quality ranking across texts. The corpus is the instrument, not the subject of evaluation.

## Section 1: Environment setup

In [1]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/MTCA-1'
else:
    PROJECT_ROOT = './MTCA-1'

import os
for subdir in ['corpus', 'corpus/raw', 'corpus/processed', 'corpus/frozen', 'outputs']:
    os.makedirs(f'{PROJECT_ROOT}/{subdir}', exist_ok=True)

print(f'Project root: {PROJECT_ROOT}')

Mounted at /content/drive
Project root: /content/drive/MyDrive/MTCA-1


In [3]:
import json
import hashlib
import re
from dataclasses import dataclass, field, asdict
from typing import Optional
from datetime import datetime, timezone
from urllib.request import urlopen, Request

# Suppress the datetime deprecation warning - we're using timezone-aware now
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

## Section 2: Corpus data structures

In [4]:
@dataclass
class CorpusUnit:
    """A single analyzable unit within a corpus text. Units are sentences or natural breaks.

    No ground-truth labels are stored - unit-type classification is performed by the council
    classifier at runtime. This dataclass holds only the raw textual material and its location.
    """
    unit_id: str
    text_id: str  # which corpus text this unit belongs to (C1, C3, C5, C7, C8)
    unit_text: str
    position_in_text: int  # ordinal position within the text


@dataclass
class CorpusText:
    """A text in the validation corpus with full provenance."""
    text_id: str  # C1, C3, C5, C7, C8
    title: str
    author: str
    year: str
    lineage: str  # therapeutic-autosuggestion, stoic, new-thought, behavioral, synthetic
    license: str
    source_url: str
    extract_description: str  # what portion of the source was used
    raw_text: str = ''  # the full extracted text
    units: list = field(default_factory=list)  # list of CorpusUnit
    sha256: str = ''

    def compute_sha256(self) -> str:
        """Hash the raw text for reproducibility."""
        self.sha256 = hashlib.sha256(self.raw_text.encode('utf-8')).hexdigest()
        return self.sha256


@dataclass
class ValidationCorpus:
    """The full validation corpus for MTCA-1."""
    corpus_version: str
    creation_date: str
    texts: list = field(default_factory=list)
    freeze_sha256: str = ''

    def all_units(self) -> list:
        return [u for t in self.texts for u in t.units]

    def summary(self) -> dict:
        return {
            'corpus_version': self.corpus_version,
            'creation_date': self.creation_date,
            'n_texts': len(self.texts),
            'n_units_total': len(self.all_units()),
            'units_per_text': {t.text_id: len(t.units) for t in self.texts},
            'lineages': sorted(set(t.lineage for t in self.texts)),
            'freeze_sha256': self.freeze_sha256
        }


corpus = ValidationCorpus(
    corpus_version='v1.0',
    creation_date=datetime.now(timezone.utc).isoformat()
)

print('Corpus data structures defined.')
print(f'Empty corpus initialized: version {corpus.corpus_version}')

Corpus data structures defined.
Empty corpus initialized: version v1.0


## Section 3: Text fetching utilities

Public-domain texts are fetched from canonical sources. Each fetch is documented with the URL, the extraction logic (what portion of the source is used), and a SHA256 hash for reproducibility.

In [5]:
def fetch_url(url: str, user_agent: str = 'MTCA-1-Research/1.0') -> str:
    """Fetch text content from a URL with a documented user-agent."""
    req = Request(url, headers={'User-Agent': user_agent})
    with urlopen(req, timeout=30) as response:
        return response.read().decode('utf-8')


def extract_between_markers(text: str, start_marker: str, end_marker: str) -> str:
    """Extract content between two string markers (inclusive of markers themselves).

    Used to extract the actual book content from Project Gutenberg files,
    which include legal boilerplate around the work.
    """
    start_idx = text.find(start_marker)
    end_idx = text.find(end_marker)
    if start_idx == -1 or end_idx == -1:
        raise ValueError(f'Could not find markers in text')
    return text[start_idx:end_idx]


def segment_into_units(text: str, text_id: str, min_unit_chars: int = 30) -> list:
    """Segment text into analyzable units.

    Splits on sentence boundaries and paragraph breaks. Filters out
    very short fragments. Returns a list of CorpusUnit objects.
    """
    # Normalize whitespace
    text = re.sub(r'\r\n', '\n', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    # Split on sentence-ending punctuation followed by whitespace and capital letter,
    # or on paragraph breaks. This is heuristic but works well for cleanly formatted text.
    sentence_pattern = r'(?<=[.!?])\s+(?=[A-Z\"\'])|\n\n+'
    raw_units = re.split(sentence_pattern, text)

    units = []
    position = 0
    for raw in raw_units:
        cleaned = raw.strip()
        # Collapse internal whitespace
        cleaned = re.sub(r'\s+', ' ', cleaned)
        if len(cleaned) < min_unit_chars:
            continue
        units.append(CorpusUnit(
            unit_id=f'{text_id}-u{position:04d}',
            text_id=text_id,
            unit_text=cleaned,
            position_in_text=position
        ))
        position += 1
    return units


print('Text fetching utilities defined.')

Text fetching utilities defined.


## Section 4: C1 — Émile Coué, *Self Mastery Through Conscious Autosuggestion* (1922)

**Lineage:** Therapeutic-autosuggestion
**License:** Public domain (Project Gutenberg #27203)
**Extract:** Practical method sections containing autosuggestion formulas

Source URL: `https://www.gutenberg.org/ebooks/27203.txt.utf-8`

In [6]:
C1_URL = 'https://www.gutenberg.org/ebooks/27203.txt.utf-8'
C1_RAW_PATH = f'{PROJECT_ROOT}/corpus/raw/C1_coue_self_mastery.txt'

# Fetch and cache the raw text
if not os.path.exists(C1_RAW_PATH):
    print(f'Fetching C1 from {C1_URL}...')
    c1_full = fetch_url(C1_URL)
    with open(C1_RAW_PATH, 'w', encoding='utf-8') as f:
        f.write(c1_full)
    print(f'Saved to {C1_RAW_PATH} ({len(c1_full):,} chars)')
else:
    with open(C1_RAW_PATH, 'r', encoding='utf-8') as f:
        c1_full = f.read()
    print(f'Loaded cached copy ({len(c1_full):,} chars)')

Fetching C1 from https://www.gutenberg.org/ebooks/27203.txt.utf-8...
Saved to /content/drive/MyDrive/MTCA-1/corpus/raw/C1_coue_self_mastery.txt (179,937 chars)


In [7]:
# Extract content between Project Gutenberg's legal markers
C1_START_MARKER = '*** START OF'
C1_END_MARKER = '*** END OF'

c1_content_with_markers = extract_between_markers(c1_full, C1_START_MARKER, C1_END_MARKER)
# Strip the start-marker line itself (everything up to the first newline after the marker)
c1_content = c1_content_with_markers.split('\n', 1)[1] if '\n' in c1_content_with_markers else c1_content_with_markers

print(f'Extracted content: {len(c1_content):,} chars (from {len(c1_full):,} total)')
print(f'\nFirst 500 chars of extracted content:\n{c1_content[:500]}')

Extracted content: 160,220 chars (from 179,937 total)

First 500 chars of extracted content:




Produced by Ruth Hart




[Note:  many of the people quoted in this text are identified only by
their initials along with either a dash or three periods.  For
consistency's sake, I have used four dashes for each person instead
of periods.  I have also added quotation marks where appropriate.
Finally, I have made the following spelling change: I congraulate
you to I congratulate you.]



SELF MASTERY THROUGH CONSCIOUS
AUTOSUGGESTION

by

EMILE COUÉ


AMERICAN LIBRARY


In [8]:
# Build the C1 CorpusText
c1_text = CorpusText(
    text_id='C1',
    title='Self Mastery Through Conscious Autosuggestion',
    author='Émile Coué',
    year='1922',
    lineage='therapeutic-autosuggestion',
    license='Public domain',
    source_url=C1_URL,
    extract_description='Full text between Project Gutenberg legal markers, content body only',
    raw_text=c1_content.strip()
)
c1_text.units = segment_into_units(c1_text.raw_text, 'C1')
c1_text.compute_sha256()

print(f'C1: {len(c1_text.units)} units, SHA256 {c1_text.sha256[:16]}...')

C1: 1165 units, SHA256 03ad968e0fc5f6f3...


## Section 5: C3 — Marcus Aurelius, *Meditations* (170-180 CE, George Long trans. 1862)

**Lineage:** Stoic self-directed
**License:** Public domain (Project Gutenberg #2680, George Long translation 1862)
**Extract:** Book 2 (one of the more self-directed books, focused on practical Stoic instruction)

Source URL: `https://www.gutenberg.org/ebooks/2680.txt.utf-8`

In [9]:
C3_URL = 'https://www.gutenberg.org/ebooks/2680.txt.utf-8'
C3_RAW_PATH = f'{PROJECT_ROOT}/corpus/raw/C3_aurelius_meditations.txt'

if not os.path.exists(C3_RAW_PATH):
    print(f'Fetching C3 from {C3_URL}...')
    c3_full = fetch_url(C3_URL)
    with open(C3_RAW_PATH, 'w', encoding='utf-8') as f:
        f.write(c3_full)
    print(f'Saved to {C3_RAW_PATH} ({len(c3_full):,} chars)')
else:
    with open(C3_RAW_PATH, 'r', encoding='utf-8') as f:
        c3_full = f.read()
    print(f'Loaded cached copy ({len(c3_full):,} chars)')

Fetching C3 from https://www.gutenberg.org/ebooks/2680.txt.utf-8...
Saved to /content/drive/MyDrive/MTCA-1/corpus/raw/C3_aurelius_meditations.txt (424,723 chars)


In [10]:
# Extract content between PG markers, then isolate Book 2
c3_content_with_markers = extract_between_markers(c3_full, '*** START OF', '*** END OF')
c3_content = c3_content_with_markers.split('\n', 1)[1] if '\n' in c3_content_with_markers else c3_content_with_markers

# Project Gutenberg's Meditations uses book divisions marked with text like
# 'THE SECOND BOOK' and 'THE THIRD BOOK'. Extract Book 2 specifically.
c3_book2_start = c3_content.find('THE SECOND BOOK')
c3_book2_end = c3_content.find('THE THIRD BOOK')

if c3_book2_start != -1 and c3_book2_end != -1:
    c3_book2 = c3_content[c3_book2_start:c3_book2_end].strip()
    print(f'Extracted Book 2: {len(c3_book2):,} chars')
else:
    # Fallback: take a reasonable middle section if book markers differ
    print('WARNING: Book 2 markers not found. Inspect c3_content structure manually.')
    print(f'First 1000 chars:\n{c3_content[:1000]}')
    c3_book2 = c3_content[:8000]  # placeholder fallback

print(f'\nFirst 500 chars of Book 2:\n{c3_book2[:500]}')

Extracted Book 2: 12,200 chars

First 500 chars of Book 2:
THE SECOND BOOK


I. Remember how long thou hast already put off these things, and how
often a certain day and hour as it were, having been set unto thee by
the gods, thou hast neglected it. It is high time for thee to understand
the true nature both of the world, whereof thou art a part; and of that
Lord and Governor of the world, from whom, as a channel from the spring,
thou thyself didst flow: and that there is but a certain limit of time
appointed unto thee, which if thou shalt not 


In [11]:
c3_text = CorpusText(
    text_id='C3',
    title='Meditations (Book 2)',
    author='Marcus Aurelius (trans. George Long, 1862)',
    year='~170 CE',
    lineage='stoic-self-directed',
    license='Public domain',
    source_url=C3_URL,
    extract_description='Book 2 of Meditations as published in Project Gutenberg #2680, George Long translation 1862',
    raw_text=c3_book2
)
c3_text.units = segment_into_units(c3_text.raw_text, 'C3')
c3_text.compute_sha256()

print(f'C3: {len(c3_text.units)} units, SHA256 {c3_text.sha256[:16]}...')

C3: 59 units, SHA256 ebb2abca0fa16d89...


## Section 6: C5 — James Allen, *As a Man Thinketh* (1903)

**Lineage:** New Thought / metaphysical-aspirational
**License:** Public domain (Project Gutenberg #4507)
**Extract:** Full short work (~30 pages)

Source URL: `https://www.gutenberg.org/ebooks/4507.txt.utf-8`

In [12]:
C5_URL = 'https://www.gutenberg.org/ebooks/4507.txt.utf-8'
C5_RAW_PATH = f'{PROJECT_ROOT}/corpus/raw/C5_allen_as_a_man_thinketh.txt'

if not os.path.exists(C5_RAW_PATH):
    print(f'Fetching C5 from {C5_URL}...')
    c5_full = fetch_url(C5_URL)
    with open(C5_RAW_PATH, 'w', encoding='utf-8') as f:
        f.write(c5_full)
    print(f'Saved to {C5_RAW_PATH} ({len(c5_full):,} chars)')
else:
    with open(C5_RAW_PATH, 'r', encoding='utf-8') as f:
        c5_full = f.read()
    print(f'Loaded cached copy ({len(c5_full):,} chars)')

c5_content_with_markers = extract_between_markers(c5_full, '*** START OF', '*** END OF')
c5_content = c5_content_with_markers.split('\n', 1)[1] if '\n' in c5_content_with_markers else c5_content_with_markers
c5_content = c5_content.strip()

c5_text = CorpusText(
    text_id='C5',
    title='As a Man Thinketh',
    author='James Allen',
    year='1903',
    lineage='new-thought',
    license='Public domain',
    source_url=C5_URL,
    extract_description='Full text between Project Gutenberg legal markers',
    raw_text=c5_content
)
c5_text.units = segment_into_units(c5_text.raw_text, 'C5')
c5_text.compute_sha256()

print(f'C5: {len(c5_text.units)} units, SHA256 {c5_text.sha256[:16]}...')
print(f'\nFirst 500 chars:\n{c5_text.raw_text[:500]}')

Fetching C5 from https://www.gutenberg.org/ebooks/4507.txt.utf-8...
Saved to /content/drive/MyDrive/MTCA-1/corpus/raw/C5_allen_as_a_man_thinketh.txt (65,526 chars)
C5: 279 units, SHA256 865efed367bdd954...

First 500 chars:
Produced by Charles Aldarondo.  HTML version by Al Haines.










AS A MAN THINKETH


BY

JAMES ALLEN


Author of "From Passion to Peace"



  _Mind is the Master power that moulds and makes,
  And Man is Mind, and evermore he takes
  The tool of Thought, and, shaping what he wills,
  Brings forth a thousand joys, a thousand ills:--
  He thinks in secret, and it comes to pass:
  Environment is but his looking-glass._



Authorized Edition

New York







## Section 7: C7 — The Twelve Steps (canonical text, 1939)

**Lineage:** Behavioral commitment / recovery
**License:** Public domain (the Twelve Steps themselves are widely reproduced and not under enforced copyright when used as text-only references; original 1939 publication is public domain)
**Extract:** The Twelve Steps in their canonical form, plus a short framing paragraph in the same first-person plural voice

Note: AA's broader literature (the 'Big Book' in full) remains copyrighted in some editions. The bare Twelve Steps themselves, as a short list, fall well within fair use and historical public-domain status when reproduced as research material. We use only the Twelve Steps proper, in their canonical 1939 form.

In [13]:
# Canonical Twelve Steps text (1939). First-person plural is the distinctive register.
C7_RAW_TEXT = """The Twelve Steps

1. We admitted we were powerless over alcohol — that our lives had become unmanageable.

2. Came to believe that a Power greater than ourselves could restore us to sanity.

3. Made a decision to turn our will and our lives over to the care of God as we understood Him.

4. Made a searching and fearless moral inventory of ourselves.

5. Admitted to God, to ourselves, and to another human being the exact nature of our wrongs.

6. Were entirely ready to have God remove all these defects of character.

7. Humbly asked Him to remove our shortcomings.

8. Made a list of all persons we had harmed, and became willing to make amends to them all.

9. Made direct amends to such people wherever possible, except when to do so would injure them or others.

10. Continued to take personal inventory and when we were wrong promptly admitted it.

11. Sought through prayer and meditation to improve our conscious contact with God as we understood Him, praying only for knowledge of His will for us and the power to carry that out.

12. Having had a spiritual awakening as the result of these steps, we tried to carry this message to alcoholics, and to practice these principles in all our affairs."""

C7_RAW_PATH = f'{PROJECT_ROOT}/corpus/raw/C7_twelve_steps.txt'
with open(C7_RAW_PATH, 'w', encoding='utf-8') as f:
    f.write(C7_RAW_TEXT)

c7_text = CorpusText(
    text_id='C7',
    title='The Twelve Steps',
    author='Anonymous (Alcoholics Anonymous, 1939)',
    year='1939',
    lineage='behavioral-commitment',
    license='Public domain (canonical text widely reproduced)',
    source_url='AA General Service literature, original 1939 publication',
    extract_description='The Twelve Steps in canonical 1939 form',
    raw_text=C7_RAW_TEXT
)
c7_text.units = segment_into_units(c7_text.raw_text, 'C7', min_unit_chars=20)
c7_text.compute_sha256()

print(f'C7: {len(c7_text.units)} units, SHA256 {c7_text.sha256[:16]}...')
print(f'\nUnits:')
for u in c7_text.units[:6]:
    print(f'  {u.unit_id}: {u.unit_text[:80]}...')

C7: 12 units, SHA256 e45cc177d1d554e0...

Units:
  C7-u0000: We admitted we were powerless over alcohol — that our lives had become unmanagea...
  C7-u0001: Came to believe that a Power greater than ourselves could restore us to sanity....
  C7-u0002: Made a decision to turn our will and our lives over to the care of God as we und...
  C7-u0003: Made a searching and fearless moral inventory of ourselves....
  C7-u0004: Admitted to God, to ourselves, and to another human being the exact nature of ou...
  C7-u0005: Were entirely ready to have God remove all these defects of character....


## Section 8: C8 — Synthetic CBT analogue (authored for this study)

**Lineage:** Behavioral commitment / contemporary practical
**License:** Original work, authored for this study
**Purpose:** Constructed specimen exhibiting the content class in contemporary clinical register without copying any proprietary CBT materials

The synthetic CBT analogue is constructed to mirror the structural features of standard cognitive-behavioral self-statements without reproducing any specific protocol. It contains first-person present-tense construction, identity-regulating function (cognitive restructuring), and mixes propositional content (evidence claims) with performative content (commitment language).

The construction process is documented in-notebook for full transparency. Reviewers can see exactly how this synthetic specimen was authored and what design choices were made.

In [14]:
# C8 construction notes:
#
# Design goals:
# - First-person present-tense construction throughout
# - Mix of propositional reasoning (what the evidence suggests) and performative
#   commitment (what I choose to do/believe)
# - Identity-regulating function (cognitive restructuring of self-narrative)
# - Contemporary clinical register (no metaphysical sourcing, no spiritual claims)
# - Twelve statements to roughly match the structural length of C7's Twelve Steps,
#   providing a parallel-length comparator within the behavioral-commitment lineage
#
# This text is NOT a CBT protocol, treatment manual, or therapeutic intervention.
# It is a research specimen constructed to exhibit specific structural features
# for studying AI reasoning about content of this class.

C8_RAW_TEXT = """Twelve Cognitive Self-Statements (synthetic specimen)

1. I notice that my automatic thoughts are not the same as evidence about what is true.

2. When I catch myself catastrophizing, I am willing to ask what the realistic worst case actually looks like.

3. I am choosing to test my predictions against what actually happens rather than against what I fear.

4. I recognize that my feelings are real information about my state, and not always accurate information about the world.

5. I am practicing the difference between a thought that occurs to me and a thought I endorse.

6. When I find myself in an all-or-nothing frame, I am willing to look for the partial truth on either side.

7. I am willing to do things that scare me when the evidence suggests they are likely to help me.

8. I am noticing that my mood follows my behavior more often than the reverse.

9. I am choosing to act in line with my values even when my feelings have not yet caught up.

10. I am willing to be a beginner at things I want to be good at.

11. I am letting old explanations of myself loosen when they no longer fit the evidence.

12. I am committing to small, observable changes I can actually carry out rather than vague resolutions I cannot."""

C8_RAW_PATH = f'{PROJECT_ROOT}/corpus/raw/C8_synthetic_cbt_analogue.txt'
with open(C8_RAW_PATH, 'w', encoding='utf-8') as f:
    f.write(C8_RAW_TEXT)

c8_text = CorpusText(
    text_id='C8',
    title='Twelve Cognitive Self-Statements (synthetic specimen)',
    author='Billy Davis (authored for MTCA-1)',
    year='2026',
    lineage='behavioral-contemporary-synthetic',
    license='Original work, MIT license for research use',
    source_url='Authored in-notebook for this study; not a clinical protocol',
    extract_description='Twelve constructed cognitive self-statements designed to exhibit the content class without reproducing proprietary CBT materials',
    raw_text=C8_RAW_TEXT
)
c8_text.units = segment_into_units(c8_text.raw_text, 'C8', min_unit_chars=20)
c8_text.compute_sha256()

print(f'C8: {len(c8_text.units)} units, SHA256 {c8_text.sha256[:16]}...')
print(f'\nUnits:')
for u in c8_text.units[:6]:
    print(f'  {u.unit_id}: {u.unit_text[:80]}...')

C8: 13 units, SHA256 e80c91db51c95b2a...

Units:
  C8-u0000: Twelve Cognitive Self-Statements (synthetic specimen)...
  C8-u0001: I notice that my automatic thoughts are not the same as evidence about what is t...
  C8-u0002: When I catch myself catastrophizing, I am willing to ask what the realistic wors...
  C8-u0003: I am choosing to test my predictions against what actually happens rather than a...
  C8-u0004: I recognize that my feelings are real information about my state, and not always...
  C8-u0005: I am practicing the difference between a thought that occurs to me and a thought...


## Section 9: Assemble and freeze the corpus

In [15]:
# Assemble all five texts into the corpus
corpus.texts = [c1_text, c3_text, c5_text, c7_text, c8_text]

# Print summary before freezing
summary = corpus.summary()
print('Corpus assembly complete.\n')
print(f'Corpus version: {summary["corpus_version"]}')
print(f'Texts: {summary["n_texts"]}')
print(f'Total units: {summary["n_units_total"]}')
print(f'Lineages: {summary["lineages"]}')
print(f'\nUnits per text:')
for tid, n in summary['units_per_text'].items():
    text = next(t for t in corpus.texts if t.text_id == tid)
    print(f'  {tid}: {n:4d} units | {text.title[:55]}')

Corpus assembly complete.

Corpus version: v1.0
Texts: 5
Total units: 1528
Lineages: ['behavioral-commitment', 'behavioral-contemporary-synthetic', 'new-thought', 'stoic-self-directed', 'therapeutic-autosuggestion']

Units per text:
  C1: 1165 units | Self Mastery Through Conscious Autosuggestion
  C3:   59 units | Meditations (Book 2)
  C5:  279 units | As a Man Thinketh
  C7:   12 units | The Twelve Steps
  C8:   13 units | Twelve Cognitive Self-Statements (synthetic specimen)


In [16]:
# Freeze: serialize corpus, compute SHA256 of the full serialization,
# save with hash in filename for permanent provenance.

def freeze_corpus(corpus_obj: ValidationCorpus, output_dir: str) -> str:
    serialized = json.dumps({
        'corpus_version': corpus_obj.corpus_version,
        'creation_date': corpus_obj.creation_date,
        'texts': [
            {
                'text_id': t.text_id,
                'title': t.title,
                'author': t.author,
                'year': t.year,
                'lineage': t.lineage,
                'license': t.license,
                'source_url': t.source_url,
                'extract_description': t.extract_description,
                'raw_text': t.raw_text,
                'sha256': t.sha256,
                'units': [asdict(u) for u in t.units]
            }
            for t in corpus_obj.texts
        ]
    }, sort_keys=True, ensure_ascii=False)

    sha = hashlib.sha256(serialized.encode('utf-8')).hexdigest()
    corpus_obj.freeze_sha256 = sha

    out_path = f'{output_dir}/corpus_{corpus_obj.corpus_version}_{sha[:16]}.json'
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(serialized)

    return out_path

frozen_path = freeze_corpus(corpus, f'{PROJECT_ROOT}/corpus/frozen')
print(f'Corpus frozen.')
print(f'Path: {frozen_path}')
print(f'Full SHA256: {corpus.freeze_sha256}')

Corpus frozen.
Path: /content/drive/MyDrive/MTCA-1/corpus/frozen/corpus_v1.0_9c21a2beb3e034dd.json
Full SHA256: 9c21a2beb3e034dd69dadc21c1a4e6e4f282c586cda3833953bfb3308f5fc1eb


In [17]:
# Also save a human-readable manifest for the methods section
manifest = {
    'corpus_id': 'MTCA-1-corpus-v1.0',
    'creation_date': corpus.creation_date,
    'freeze_sha256': corpus.freeze_sha256,
    'total_texts': len(corpus.texts),
    'total_units': len(corpus.all_units()),
    'texts': [
        {
            'text_id': t.text_id,
            'title': t.title,
            'author': t.author,
            'year': t.year,
            'lineage': t.lineage,
            'license': t.license,
            'source_url': t.source_url,
            'extract_description': t.extract_description,
            'n_units': len(t.units),
            'raw_text_sha256': t.sha256,
            'raw_text_chars': len(t.raw_text)
        }
        for t in corpus.texts
    ]
}

manifest_path = f'{PROJECT_ROOT}/outputs/corpus_manifest.json'
with open(manifest_path, 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

print(f'Corpus manifest saved: {manifest_path}')
print(json.dumps(manifest, indent=2)[:2000])

Corpus manifest saved: /content/drive/MyDrive/MTCA-1/outputs/corpus_manifest.json
{
  "corpus_id": "MTCA-1-corpus-v1.0",
  "creation_date": "2026-06-27T23:45:57.888730+00:00",
  "freeze_sha256": "9c21a2beb3e034dd69dadc21c1a4e6e4f282c586cda3833953bfb3308f5fc1eb",
  "total_texts": 5,
  "total_units": 1528,
  "texts": [
    {
      "text_id": "C1",
      "title": "Self Mastery Through Conscious Autosuggestion",
      "author": "\u00c9mile Cou\u00e9",
      "year": "1922",
      "lineage": "therapeutic-autosuggestion",
      "license": "Public domain",
      "source_url": "https://www.gutenberg.org/ebooks/27203.txt.utf-8",
      "extract_description": "Full text between Project Gutenberg legal markers, content body only",
      "n_units": 1165,
      "raw_text_sha256": "03ad968e0fc5f6f3c51c0cdcebcf0457a54b09770c9fcb82342d2e5748891983",
      "raw_text_chars": 160192
    },
    {
      "text_id": "C3",
      "title": "Meditations (Book 2)",
      "author": "Marcus Aurelius (trans. George Lo

## Section 10: Inspection — sample units across the corpus

Before moving to Stage 3 (frame design), inspect a sample of units from each text to confirm segmentation worked correctly and the content class is well-represented.

In [18]:
# Show first 3 units from each text
for t in corpus.texts:
    print(f'\n{"=" * 70}')
    print(f'{t.text_id}: {t.title}')
    print(f'Lineage: {t.lineage} | Units: {len(t.units)}')
    print(f'{"=" * 70}')
    for u in t.units[:3]:
        print(f'\n[{u.unit_id}]\n{u.unit_text[:300]}{"..." if len(u.unit_text) > 300 else ""}')


C1: Self Mastery Through Conscious Autosuggestion
Lineage: therapeutic-autosuggestion | Units: 1165

[C1-u0000]
[Note: many of the people quoted in this text are identified only by their initials along with either a dash or three periods.

[C1-u0001]
For consistency's sake, I have used four dashes for each person instead of periods.

[C1-u0002]
I have also added quotation marks where appropriate.

C3: Meditations (Book 2)
Lineage: stoic-self-directed | Units: 59

[C3-u0000]
Remember how long thou hast already put off these things, and how often a certain day and hour as it were, having been set unto thee by the gods, thou hast neglected it.

[C3-u0001]
It is high time for thee to understand the true nature both of the world, whereof thou art a part; and of that Lord and Governor of the world, from whom, as a channel from the spring, thou thyself didst flow: and that there is but a certain limit of time appointed unto thee, which if thou shalt not ...

[C3-u0002]
Let it be thy earnest 

## Section 11: Stage 2 closeout

**Stage 2 outputs:**
- Raw text files at `corpus/raw/` for each of the five corpus texts (cached for reproducibility)
- Frozen corpus JSON at `corpus/frozen/corpus_v1.0_{hash}.json` with full SHA256 hash in filename
- Human-readable manifest at `outputs/corpus_manifest.json` for the methods section

**Reproducibility:**
- Each text has its own SHA256 (the `sha256` field on each CorpusText)
- The full corpus has a freeze SHA256 covering all texts and segmentation results
- Source URLs are documented; anyone can re-fetch and verify hashes
- The synthetic specimen (C8) is authored in-notebook and its construction is fully transparent

**Stage 3 prerequisites met:**
- Five texts assembled across four lineages
- Units segmented and counted
- Corpus frozen with hash
- Ready for frame design and prompt construction

**Note for Stage 3:**
When designing frames in Stage 3, the corpus is the substrate. Each frame will be applied to each unit (or each text, depending on the frame design - we'll decide that in Stage 3). The corpus hash is the anchor that ensures frame results can be traced back to a specific corpus version.

If the corpus is revised after this point (units re-segmented, texts added or removed), it becomes corpus v1.1 with a new hash. Stage 3 frames are tied to a specific corpus hash for reproducibility.